In [ ]:
from rsa import *
import rsa

In [ ]:
rsa.PrivateKey?

In [ ]:
dir()

In [ ]:
t = rsa.PrivateKey.load_pkcs1("<REDACTED:rsa-private-key>")

In [ ]:
rsa.PublicKey.load_pkcs1("<REDACTED:rsa-public-key>")

In [ ]:
from torch.nn import *
import torch
import numpy as np

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def pile(a):
    t = [torch.ones(()).broadcast_to(*a.shape[:-2], a.size(-1))]
    for i in range(a.size(-2) - 1):
        i = (t[-1] * a[..., i, :]).cumsum_(-1)
        t.append(i / i[..., -1:])
    return torch.stack(t, -2)


def piles(a):
    t = pile(a)
    t *= pile(a.flip(-2, -1)).flip(-2, -1)
    t *= a
    return t


def crush(a, i, j=None):
    if j is None:
        return a.view(*a.shape[:i], -1, *a.shape[i % a.ndim + 2 :])
    return a.view(*a.shape[:i], a.shape[i : i + 2].numel(), *a.shape[i + 2 : j], -1, *a.shape[j % a.ndim + 2 :])


def uncrush(a, i, v):
    return a.view(*a.shape[:i], v, -1, *a.shape[i % a.ndim + 1 :])


class F(Module):
    def __init__(s, f, a=None):
        super().__init__()
        s.f = f
        s.a = a

    def forward(s, x):
        return s.f(x, *() if s.a is None else (s.a,))

    def extra_repr(s):
        return f"{s.f}, {s.a}"


class Linears(Module):
    def __new__(s, i, o, n=1, b=1, dim=0):
        if n < 2:
            print(end="Linear:downgrade ")
            return Linear(i, o, b)
        return super().__new__(s)

    def __init__(s, i, o, n=1, b=None, dim=0):
        super().__init__()
        s.w = Parameter(init.kaiming_uniform_(torch.empty(n, *(1,) * dim, o, i)).transpose(-2, -1))
        s.b = Parameter(init.uniform_(torch.empty(n, *(1,) * dim, 1, o), -(i ** 0.5), i ** 0.5)) if b else None

    def forward(s, x):
        x = x @ s.w
        return x if s.b is None else x + s.b

    def extra_repr(s):
        return f"{s.w.size(1)}, {s.w.size(2)}, {s.w.size(0)}, {s.b is not None}"


class Map(Sequential):
    def forward(s, x):
        return [s(x) for s, x in zip(s, x)]


def Score(q, k, h, n=1):
    return Sequential(
        Map(Linears(q, h, n), Linears(k, h, n, 0)),
        F(lambda x: x[0][..., None, :] + x[1][..., None, :, :]),
        Hardswish(1),
        Linears(h, 1, n, 1, 1),
        F(lambda x: piles(x[..., 0].softmax(-1))),
    )


def Attention(q, k, v, h, o, n=1):
    return Sequential(
        Map(Score(q, k, h, n), Linears(v, h, n)),
        F(lambda x: x[0] @ x[1]),
        F(lambda x: x.sum(-3) if x.ndim > 2 else x),
        Linear(h, o),
    )


In [ ]:
from PIL.Image import open

a = torch.tensor(np.array(open("assets/wallpaper2.png").resize((48, 27)), dtype="f")[..., :3])
b = a.mean(2, keepdim=True)
h = w = 4
m = Attention(1, 3, 3, 4, 3)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.01, 10000, pct_start=0.01, div_factor=9, final_div_factor=99)
for i in range(O.total_steps // 500):
    for i in range(100):
        i = np.random.randint(0, len(a) - h + 1, 128)
        j = np.random.randint(0, a.shape[1] - w + 1, 128)
        i = functional.mse_loss(
            m(((torch.stack([b[i : i + h, j : j + w] for i, j in zip(i, j)])[:, None], a), a)),
            torch.stack([a[i : i + h, j : j + w] for i, j in zip(i, j)]),
        )
        i.backward()
        o.step()
        O.step()
        o.zero_grad()
    print(i)